# 25. 최종 모델 평가 (Final Model Evaluation)

목적: 이전 단계(`07_01_evaluation.ipynb`)에서 산출된 최적 임계값을 불러와, 테스트 데이터(`test.parquet`)에 대한 행 단위(Row-level) 및 개체 단위(Entity-level) 최종 성능 평가를 수행합니다.

In [ ]:
import sys, os, json, joblib
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("..\\"))

from src.eval_core import evaluate_row_level, EntityLevelEvaluator
import config.eval_config as cfg
import config.train_config as tcfg
print("환경 준비 완료")

## 1. 앙상블 모델 및 최적 임계값 로드


In [ ]:
from pathlib import Path
SAVE_DIR = Path(tcfg.MODEL_SAVE_DIR)

with open(SAVE_DIR / "feature_cols.json", encoding="utf-8") as f:
    FEATURE_COLS = json.load(f)

with open(SAVE_DIR / "best_threshold.json", encoding="utf-8") as f:
    THRESHOLD = json.load(f)["threshold"]

models = [joblib.load(p) for p in sorted(SAVE_DIR.glob("subset_*.pkl"))]
print(f"로드: {len(models)}개 서브셋 모델, 피처: {len(FEATURE_COLS)}개, 임계값: {THRESHOLD:.4f}")

class _EnsembleInfer:
    def __init__(self, models): self.models = models
    def predict_proba(self, df, feature_cols):
        return np.mean([m.predict_proba(df[feature_cols])[:,1] for m in self.models], axis=0)

ensemble = _EnsembleInfer(models)

## 2. 행 단위 평가 (Row-level Evaluation)


In [ ]:
df_test = pd.read_parquet(cfg.TEST_PATH)
y_test_prob = ensemble.predict_proba(df_test, FEATURE_COLS)

row_result = evaluate_row_level(
    df_test[cfg.TARGET_COL].values,
    y_test_prob,
    THRESHOLD,
    title="Test Set",
)

## 3. 개체 단위 롤링 평가 (Entity-level Evaluation)


In [ ]:
ev = EntityLevelEvaluator(
    serial_col=cfg.SERIAL_COL,
    date_col=cfg.DATE_COL,
    target_col=cfg.TARGET_COL,
)
entity_result = ev.evaluate(df_test, y_test_prob, THRESHOLD)
EntityLevelEvaluator.plot(entity_result)

## 4. 미탐 개체 상세 확인


In [ ]:
ed = entity_result["entity_df"]
miss_list = ed[(ed["is_failure"]==1) & (ed["has_alarm"]==0)]
print(f"미탐 고장 개체 수: {len(miss_list)}")
miss_list.head(20)